In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [ ]:
files

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
len(documents)

72

In [5]:
from minsearch import Index
def build_index(documents):
    index = Index(
        text_fields = ["content"],
        keyword_fields = ["filename"]
    )
    index.fit(documents)    
    return index

In [ ]:
index = build_index(documents)
index.search("How does the agentic loop keep calling the model until it stops?" , num_results=3)

In [7]:
from rag_helper import RAGBase
rag_helper = RAGBase(index = index)

In [8]:
response , answer = rag_helper.rag("How does the agentic loop keep calling the model until it stops?")

In [9]:
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""The agentic loop keeps calling the model until it stops by continuously checking the model's response for function calls. Here's how it works:

1.  **Initial Call:** The loop starts by sending the user's question and the `developer` instructions (which define the agent's role and tools) to the model.
2.  **Model's Decision:** The model processes this input and decides its next action.
    *   **If it needs a tool:** The model returns a `function_call` in its response, specifying the tool to be called (e.g., `search`) and its arguments.
    *   **If it has a final answer:** The model returns a `message` with the answer and no function calls.
3.  **Executing Tools (if any):** If the model's response contains `function_call` entries:
    *   The code executes each requested function (e.g., `make_call` for the `search` tool).
    *   The output of thes

In [10]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [11]:
len(chunks)

295

In [ ]:
index = build_index(chunks)
index.search("How does the agentic loop keep calling the model until it stops?" , num_results=3)

In [14]:
from rag_helper import RAGBase
rag_helper = RAGBase(index = index)

In [17]:
response , answer = rag_helper.rag("How does the agentic loop keep calling the model until it stops?")

In [18]:
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""The agentic loop keeps calling the model using a `while True` loop.

Here's how it works:
1.  **Initialize Flag**: At the beginning of each iteration, a flag `has_function_calls` is set to `False`.
2.  **Call Model**: The loop calls the model (e.g., `openai_client.responses.create`) with the current message history.
3.  **Check for Function Calls**: It then iterates through the model's output.
    *   If an item of `type == "function_call"` is found, it means the model wants to use a tool. The function call is executed, its output is added to the message history, and `has_function_calls` is set to `True`.
    *   If an item of `type == "message"` is found, it's considered part of the assistant's response.
4.  **Loop Condition**: After processing all items in the model's output for the current turn, the loop checks the `has_function_calls` flag.
   

In [19]:
def search(query: str) -> dict[str, str]:
    """
    Search the lessons database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
    )

In [21]:
from openai import OpenAI

from toyaikit.llm import OpenAIChatCompletionsClient , OpenAIClient
from toyaikit.chat.runners import OpenAIChatCompletionsRunner
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback, OpenAIResponsesRunner

In [22]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [23]:

agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the lessons database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [24]:
from dotenv import load_dotenv
import os
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [25]:
#we use the openAi format and syntax , but the backend is calling the gemini api, since the toyaikit library is built for openai api
gemini_client = OpenAI(
    api_key= GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [27]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
"""

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)
runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIChatCompletionsClient(
        model="gemini-2.5-flash-lite",
        client=gemini_client,
    )
)

In [ ]:
resultat = runner.loop(
    prompt= "How does the agentic loop work, and how is it different from plain RAG?",
    callback = callback
)